In [1]:
"""
FinDER (FinanceRAG) retrieval evaluation harness.
 
Loads the FinDER sub-task of Linq-AI-Research/FinanceRAG (BEIR-style
corpus/queries/qrels), chunks the corpus with several character-based
splitter configurations, encodes chunks + queries with a
SentenceTransformer, retrieves top-k chunks per query with cosine
similarity, and scores each splitter config against the qrels with
Recall@k and NDCG@10.
 
Usage:
    pip install sentence-transformers datasets scikit-learn numpy pandas --break-system-packages
    python finder_splitter_eval.py
 
Notes on data format (per the dataset card / BEIR convention):
    - corpus split: {_id, title, text}
    - queries split: {_id, text}
    - qrels split: query-id, corpus-id, score  (relevance judgments)
If the qrels split name or column names differ slightly in a given
dataset revision, adjust QRELS_COLS / QRELS_SPLIT_NAME below — the
Hugging Face card does not pin an exact split name, so this is the
one part of the script most likely to need a small tweak.
"""

'\nFinDER (FinanceRAG) retrieval evaluation harness.\n\nLoads the FinDER sub-task of Linq-AI-Research/FinanceRAG (BEIR-style\ncorpus/queries/qrels), chunks the corpus with several character-based\nsplitter configurations, encodes chunks + queries with a\nSentenceTransformer, retrieves top-k chunks per query with cosine\nsimilarity, and scores each splitter config against the qrels with\nRecall@k and NDCG@10.\n\nUsage:\n    pip install sentence-transformers datasets scikit-learn numpy pandas --break-system-packages\n    python finder_splitter_eval.py\n\nNotes on data format (per the dataset card / BEIR convention):\n    - corpus split: {_id, title, text}\n    - queries split: {_id, text}\n    - qrels split: query-id, corpus-id, score  (relevance judgments)\nIf the qrels split name or column names differ slightly in a given\ndataset revision, adjust QRELS_COLS / QRELS_SPLIT_NAME below — the\nHugging Face card does not pin an exact split name, so this is the\none part of the script most l

In [2]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    

PyTorch version: 2.5.1
CUDA available: True
GPU: NVIDIA GeForce RTX 2070 with Max-Q Design


In [3]:

device = "cuda" if torch.cuda.is_available() else "cpu"

print( device)

cuda


In [4]:
from llama_index.core.node_parser import SentenceSplitter

D:\conda_envs\finder\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [5]:
#!pip install langchain-text-splitters


In [6]:
import math                                   # for log2() used in the DCG/NDCG formula
import itertools                              # imported for convenience; not directly used yet, safe to remove if unused
from dataclasses import dataclass             # lets us define a lightweight Chunk record without boilerplate __init__
 
import numpy as np                            # vector math: embeddings, cosine similarity, argpartition/argsort
import pandas as pd                           # builds and prints/saves the final comparison table
from datasets import load_dataset             # Hugging Face loader for the FinanceRAG corpus/queries/qrels splits
from sentence_transformers import SentenceTransformer  #
from sklearn.metrics.pairwise import cosine_similarity
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter
)

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"


D:\conda_envs\finder\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
 
DATASET_NAME = "Linq-AI-Research/FinanceRAG"  # HF dataset repo id
SUBTASK = "FinDER"                            # which FinanceRAG sub-task/config to load
QRELS_SPLIT_NAME = "qrels"                    # name of the relevance-judgments split; adjust if wrong on HF
#EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # embedding model id; swap for a finance-tuned model if you have one
TOP_K = 10                                    # how many chunks to retrieve per query, and the k in Recall@k/NDCG@k
MAX_QUERIES = 200                             # cap on number of queries for a fast first pass; None = use all queries
MAX_CORPUS_DOCS = None                        # cap on number of corpus docs for a fast first pass; None = use full corpus

QRELS_COLS = ("query_id", "corpus_id", "score")
EMBED_MODEL = "BAAI/bge-base-en-v1.5"


In [8]:
# Character-splitter configs to compare: (chunk_size, chunk_overlap)
SPLITTER_CONFIGS = [
    (256, 0),                                 # small chunks, no overlap
    (256, 64),                                # small chunks, some overlap to avoid cutting context at boundaries
    (512, 0),                                 # medium chunks, no overlap
    (512, 100),                               # medium chunks, moderate overlap
    (1024, 0),                                # large chunks, no overlap
    (1024, 150),                              # large chunks, larger overlap
]


In [9]:
# ---------------------------------------------------------------------------
# Character splitter (plain character-window splitter — no token/sentence
# awareness by design, to match "character splitter" evaluation).
# ---------------------------------------------------------------------------
 
def char_split(text: str, chunk_size: int, chunk_overlap: int) -> list[str]:
    # function signature: takes raw text and returns a list of character-window chunks
    if chunk_overlap >= chunk_size:
        # overlap must be strictly smaller than chunk_size or the sliding window never advances
        raise ValueError("chunk_overlap must be smaller than chunk_size")
    if not text:
        # nothing to split if the text is empty/None
        return []
    step = chunk_size - chunk_overlap         # how far the window slides forward each iteration
    chunks = []                                # accumulator for the resulting chunk strings
    for start in range(0, len(text), step):    # slide a window across the text in `step`-sized increments
        chunk = text[start:start + chunk_size]  # slice out one chunk_size-character window
        if chunk.strip():                       # skip chunks that are empty/whitespace-only
            chunks.append(chunk)                # keep the non-empty chunk
        if start + chunk_size >= len(text):     # if this window already reached the end of the text
            break                               # stop looping — no more text left to cover
    return chunks
    

In [10]:
def get_splitter(
    splitter_name,
    chunk_size,
    chunk_overlap
):

    if splitter_name == "llama_sentence":

        return SentenceSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

    elif splitter_name == "langchain_character":

        return CharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

    elif splitter_name == "langchain_recursive":

        return RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

    else:
        raise ValueError(
            f"Unknown splitter: {splitter_name}"
        )

In [11]:
@dataclass
class Chunk:
    # simple container pairing a chunk of text with the id of the document it came from
    doc_id: str                                 # source document's _id, needed later to score against qrels
    text: str                                   # the actual chunk text that gets embedded
 

In [12]:
def build_chunks(corpus_rows, chunk_size: int, chunk_overlap: int) -> list[Chunk]:
    # turns every corpus document into a list of Chunk objects using the given splitter config
    chunks = []                                 # accumulator across the whole corpus
    for row in corpus_rows:                     # iterate over every document in the corpus
        title = (row.get("title") or "").strip()  # pull the title field, defaulting to "" if missing
        body = row.get("text") or ""             # pull the body text field, defaulting to "" if missing
        full_text = f"{title}\n{body}" if title else body  # prepend title to body when a title exists
        for piece in char_split(full_text, chunk_size, chunk_overlap):  # split this doc's text into windows
            chunks.append(Chunk(doc_id=row["_id"], text=piece))  # record each chunk with its source doc id
    return chunks

def chunk_documents(
    documents,
    splitter_type="recursive",
    chunk_size=1000,
    chunk_overlap=200
):

    splitter = create_splitter(
        splitter_type,
        chunk_size,
        chunk_overlap
    )

    chunks = []

    for doc in documents:

        doc_id = doc["id"]
        text = doc["text"]

        doc_chunks = splitter.split_text(text)

        for i, chunk in enumerate(doc_chunks):

            chunks.append({
                "chunk_id": f"{doc_id}_{i}",
                "document_id": doc_id,
                "text": chunk
            })

    return chunks

In [13]:

def dcg(rels: list[int]) -> float:
    # discounted cumulative gain: sum of relevance / log2(rank+1), rewarding relevant hits ranked higher
    return sum(r / math.log2(i + 2) for i, r in enumerate(rels))
    

In [14]:

def recall_at_k(retrieved_doc_ids: list[str], relevant_doc_ids: set[str], k: int) -> float:
    # fraction of all relevant docs for this query that showed up in the top-k retrieved results
    if not relevant_doc_ids:
        # no known relevant docs for this query means recall is undefined here; return 0 rather than error
        return 0.0
    hit = len(set(retrieved_doc_ids[:k]) & relevant_doc_ids)  # count overlap between top-k retrieved and truly relevant docs
    return hit / len(relevant_doc_ids)
    

In [15]:

def ndcg_at_k(retrieved_doc_ids: list[str], relevant_doc_ids: set[str], k: int) -> float:
    # normalized DCG: actual DCG divided by the best-possible (ideal) DCG for this query
    rels = [1 if d in relevant_doc_ids else 0 for d in retrieved_doc_ids[:k]]  # binary relevance per retrieved doc, top-k only
    ideal = sorted(rels, reverse=True)            # the best possible ordering of those same relevance labels
    ideal_dcg = dcg(ideal)                        # DCG of that ideal ordering — the normalizing denominator
    return dcg(rels) / ideal_dcg if ideal_dcg > 0 else 0.0  # avoid divide-by-zero when there's no relevant doc at all




In [16]:
from datasets import load_dataset

CORPUS_PATH = "../finrag_data/finder_corpus.jsonl/corpus.jsonl"    # your downloaded corpus file
QUERIES_PATH = "../finrag_data/finder_queries.jsonl/queries.jsonl"  # your downloaded queries file
QRELS_PATH = "../finrag_data/FinDER_qrels.tsv"        # point this at wherever your qrels/labels file landed

from sklearn.metrics.pairwise import cosine_similarity

top_k = 5

def main():
    print("Loading {DATASET_NAME} / {SUBTASK}...")
    # load all documents to be searched
    corpus_ds = load_dataset("json", data_files=CORPUS_PATH, split="train")
    # load all search queries
    queries_ds = load_dataset("json", data_files=QUERIES_PATH, split="train")
    try:
        qrels_ds = load_dataset("csv", data_files=QRELS_PATH, delimiter="\t", split="train")
    except Exception as e:
        raise SystemExit(
            f"Could not load qrels file at '{QRELS_PATH}'. Check the file exists and "
            f"that QRELS_COLS matches its actual column names.\nOriginal error: {e}"
        )

    corpus_rows = list(corpus_ds)
    if MAX_CORPUS_DOCS:
        corpus_rows = corpus_rows[:MAX_CORPUS_DOCS]

    queries_rows = list(queries_ds)
    query_ids = { r["_id"] for r in queries_rows}
    
    print(corpus_rows[:5])
    print(queries_rows[:5])

    #QRELS_COLS = ("query-id", "corpus-id", "score")
    qcol, dcol, scol = QRELS_COLS
    relevant = {}
    for row in qrels_ds:
        qid = row[qcol]
        score = row["score"]
        if qid not in query_ids:
            continue
        if row.get(score, 1) and  int(row[scol]) > 0:
            relevant.setdefault(qid, set()).add(row[dcol])

    print( f"Corpus docs: {len(corpus_rows)} | Queries: {len(queries_rows)}"
           f" Queries with qrels: {len(relevant)}")

    model = SentenceTransformer(EMBED_MODEL,
                                device=device
                                )

    query_texts = [r["text"] for r in queries_rows]
    query_vecs = model.encode(query_texts, normalize_embeddings=True, show_progress_bar=True)

    chunk_size = 256
    chunk_overlap = 64
    splitter = get_splitter("llama_sentence", 256, 64)
    chunks = []
    chunk_texts = []
    chunk_doc_ids = []
    for corpus in corpus_rows:
        doc_id = corpus["_id"]
        text = corpus["text"]
        
        corpus_chunks = splitter.split_text(text)

        for i, chunk in enumerate(corpus_chunks):
            chunks.append({
                "chunk_id":f"{doc_id}_{i}",
                "document_id":doc_id,
                "text":chunk
            })
    
    
        chunk_texts = [c["text"] for c in chunks]
        chunk_doc_ids = [c["document_id"] for c in chunks]

    chunk_vecs = model.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=True)
    scores_matrix = cosine_similarity( query_vecs, chunk_vecs)
    #top_indices = np.argsort( scores )[::-1][:top_k]

    recalls, ndcgs = [], []
    results = []
    for qi, row in enumerate(queries_rows):
        qid = row["_id"]
        #score = row["score"]
        rel_docs = relevant.get(qid)
        if not rel_docs:
            continue
        query_scores = scores_matrix[qi]                               # this query's row only
        top_indices = np.argsort(query_scores)[::-1][:top_k]  
        retrieved_doc_ids = [chunk_doc_ids[i] for i in top_indices]
        recalls.append(recall_at_k(retrieved_doc_ids, rel_docs, top_k))
        ndcgs.append(ndcg_at_k(retrieved_doc_ids, rel_docs, top_k))

    results.append({                                      # summarize this splitter config's performance
            "chunk_size": chunk_size,                          # record the config's chunk size
            "chunk_overlap": chunk_overlap,                    # record the config's overlap
            "n_chunks": len(chunks),                           # how many chunks this config produced overall
            "n_queries_scored": len(recalls),                  # how many queries actually contributed to the averages
            f"recall@{top_k}": np.mean(recalls) if recalls else float("nan"),  # mean recall across scored queries
            f"ndcg@{top_k}": np.mean(ndcgs) if ndcgs else float("nan"),        # mean NDCG across scored queries
        }) 
    df = pd.DataFrame(results).sort_values(f"ndcg@{top_k}", ascending=False)  # tabulate results, best NDCG first
    print("\n=== Splitter comparison (sorted by NDCG@%d) ===" % top_k)  # header before printing the table
    print(df.to_string(index=False))                           # print the full comparison table to stdout
 
    out_path = "../results/finder_splitter_baseresults.csv"                    # where to persist the results
    df.to_csv(out_path, index=False)                             # write the comparison table to a CSV file
    print(f"\nSaved results to {out_path}") 



In [17]:
if __name__ == "__main__":
    # standard Python entry-point guard: only run main() when this file is executed directly, not when imported
    main()

Loading {DATASET_NAME} / {SUBTASK}...


Generating train split: 13867 examples [00:00, 187739.99 examples/s]
Generating train split: 216 examples [00:00, 10719.76 examples/s]
Generating train split: 103 examples [00:00, 15437.87 examples/s]


[{'_id': 'ADBE20230004', 'title': 'ADBE OVERVIEW', 'text': 'Adobe is a global technology company with a mission to change the world through personalized digital experiences. For over four decades, Adobe’s innovations have transformed how individuals, teams, businesses, enterprises, institutions, and governments engage and interact across all types of media. Our products, services and solutions are used around the world to imagine, create, manage, deliver, measure, optimize and engage with content across surfaces and fuel digital experiences. We have a diverse user base that includes consumers, communicators, creative professionals, developers, students, small and medium businesses and enterprises. We are also empowering creators by putting the power of artificial intelligence (“AI”) in their hands, and doing so in ways we believe are responsible. Our products and services help unleash creativity, accelerate document productivity and power businesses in a digital world.'}, {'_id': 'ADBE

'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].
'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/adapter_config.json
Retrying in 2s [Retry 2/5].
'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/adapter_config.json
Retrying in 4s [Retry 3/5].
Batches: 100%|██████████| 488/488 [04:48<00:00,  1.69it/s]



=== Splitter comparison (sorted by NDCG@5) ===
 chunk_size  chunk_overlap  n_chunks  n_queries_scored  recall@5   ndcg@5
        256             64     15594                64  0.351562 0.280555

Saved results to finder_splitter_results.csv
